In [2]:
import os
import re
import cv2
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ---- CONFIG ----
original_dir = "/content/drive/MyDrive/brown_spot"       # base leaf images (BROWNSPOT1_092.jpg etc.)
synthetic_dir = "/content/drive/MyDrive/blast_on_brown_spot_cleaned" # e.g. blast_on_brownspot_BROWNSPOT1_092.jpg
output_csv = "/content/drive/MyDrive/psnr_validation_report.csv"
psnr_threshold = 25.0

# Pattern: {lesion}_on_{baseclass}_{rest_of_original_filename}
pattern = re.compile(r"^[a-zA-Z]+_on_[a-zA-Z]+_(.+)$")

def get_base_filename(syn_fname):
    match = pattern.match(syn_fname)
    if match:
        return match.group(1)  # e.g. "BROWNSPOT1_092.jpg"
    return None

# ---- VALIDATION ----
results = []
synthetic_files = sorted(os.listdir(synthetic_dir))

for fname in synthetic_files:
    syn_path = os.path.join(synthetic_dir, fname)
    base_fname = get_base_filename(fname)

    if base_fname is None:
        results.append({"filename": fname, "status": "pattern_mismatch",
                         "matched_base": None, "psnr": None, "ssim": None})
        continue

    orig_path = os.path.join(original_dir, base_fname)

    if not os.path.exists(orig_path):
        results.append({"filename": fname, "status": "missing_reference",
                         "matched_base": base_fname, "psnr": None, "ssim": None})
        continue

    orig = cv2.imread(orig_path)
    syn = cv2.imread(syn_path)

    if orig is None or syn is None:
        results.append({"filename": fname, "status": "read_error",
                         "matched_base": base_fname, "psnr": None, "ssim": None})
        continue

    if orig.shape != syn.shape:
        syn = cv2.resize(syn, (orig.shape[1], orig.shape[0]))

    orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    syn_rgb = cv2.cvtColor(syn, cv2.COLOR_BGR2RGB)

    p_val = psnr(orig_rgb, syn_rgb, data_range=255)
    s_val = ssim(orig_rgb, syn_rgb, channel_axis=2, data_range=255)

    status = "ok" if p_val >= psnr_threshold else "low_quality"

    results.append({
        "filename": fname,
        "status": status,
        "matched_base": base_fname,
        "psnr": round(p_val, 2),
        "ssim": round(s_val, 4)
    })

# ---- SUMMARY ----
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)

valid = df.dropna(subset=["psnr"])
print(f"Total images checked: {len(df)}")
print(f"Valid comparisons: {len(valid)}")
print(f"Pattern mismatches: {(df['status']=='pattern_mismatch').sum()}")
print(f"Missing references: {(df['status']=='missing_reference').sum()}")
print(f"Read errors: {(df['status']=='read_error').sum()}")
if len(valid) > 0:
    print(f"Mean PSNR: {valid['psnr'].mean():.2f} dB")
    print(f"Std PSNR:  {valid['psnr'].std():.2f} dB")
    print(f"Min PSNR:  {valid['psnr'].min():.2f} dB")
    print(f"Max PSNR:  {valid['psnr'].max():.2f} dB")
    print(f"Flagged low quality (< {psnr_threshold} dB): {(valid['status']=='low_quality').sum()}")

df.head(10)

df.head(10)

Total images checked: 618
Valid comparisons: 618
Pattern mismatches: 0
Missing references: 0
Read errors: 0
Mean PSNR: 39.92 dB
Std PSNR:  4.88 dB
Min PSNR:  20.61 dB
Max PSNR:  60.31 dB
Flagged low quality (< 25.0 dB): 1


,filename,status,matched_base,psnr,ssim
0,blast_on_brownspot_BROWNSPOT1_092.jpg,ok,BROWNSPOT1_092.jpg,30.74,0.9622
1,blast_on_brownspot_BROWNSPOT3_036.jpg,ok,BROWNSPOT3_036.jpg,35.17,0.9665
2,blast_on_brownspot_BROWNSPOT3_068.jpg,ok,BROWNSPOT3_068.jpg,31.21,0.9641
3,blast_on_brownspot_BROWNSPOT3_095.jpg,ok,BROWNSPOT3_095.jpg,29.52,0.9581
4,blast_on_brownspot_BROWNSPOT4_020.jpg,ok,BROWNSPOT4_020.jpg,36.43,0.9752
5,blast_on_brownspot_BROWNSPOT4_057.jpg,ok,BROWNSPOT4_057.jpg,25.98,0.9466
6,blast_on_brownspot_BROWNSPOT4_058.jpg,ok,BROWNSPOT4_058.jpg,33.90,0.9714
7,blast_on_brownspot_BROWNSPOT4_075.jpg,ok,BROWNSPOT4_075.jpg,35.99,0.9748
8,blast_on_brownspot_BROWNSPOT4_095.jpg,ok,BROWNSPOT4_095.jpg,33.88,0.9650
9,blast_on_brownspot_BROWNSPOT4_167.jpg,ok,BROWNSPOT4_167.jpg,41.49,0.9776
